This script selects ATKIS polygons in which the entire built-up area consists of buildings constructed after 2015. Adjacent polygons are dissolved, areas smaller than 1 hectare are removed, and the remaining new housing development areas are exported as a GeoPackage.

In [ ]:
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.mask import mask as rio_mask
from pathlib import Path

# =============================================================================
# CONFIGURATION
# =============================================================================

INPUT_GPKG = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\New_Housing_Development_Areas\New_Housing_Development_Areas_buildup_ratio.gpkg"
OUTPUT_DIR = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\New_Housing_Development_Areas"

# Filter: retain only polygons where the entire built-up area consists of new buildings
THRESHOLD_RATIO_NEW_VS_ALL = 1.0

# =============================================================================
# MAIN
# =============================================================================

def main():
    output_path = Path(OUTPUT_DIR)
    output_path.mkdir(parents=True, exist_ok=True)

    # =========================================================================
    # STEP 1: Load data
    # =========================================================================
    print("Loading ATKIS data...")
    gdf = gpd.read_file(INPUT_GPKG)
    print(f"✓ {len(gdf):,} polygons (CRS: {gdf.crs})")

    # =========================================================================
    # STEP 2: Filter – retain only completely new developments
    #          (ratio_new_vs_all == 1.0)
    # =========================================================================
    mask = gdf["ratio_new_vs_all"] >= THRESHOLD_RATIO_NEW_VS_ALL
    gdf_filtered = gdf[mask].copy().reset_index(drop=True)
    n_in = len(gdf_filtered)

    # =========================================================================
    # STEP 3: Dissolve adjacent polygons and apply minimum area threshold
    # =========================================================================
    print("\nDissolving adjacent polygons...")
    dissolved = (
        gdf_filtered
        .dissolve()
        .explode(index_parts=False)
        .reset_index(drop=True)
    )

    print(f"-> Before dissolve:  {len(gdf_filtered):,} polygons")
    print(f"-> After dissolve:   {len(dissolved):,} connected areas")
    print(f"-> Merged:           {len(gdf_filtered) - len(dissolved):,} polygons")

    dissolved["area_ha"] = dissolved.geometry.area / 10000
    n_before = len(dissolved)

    dissolved = (
        dissolved[dissolved["area_ha"] >= 1.0]
        .reset_index(drop=True)
    )

    print(
        f"-> After minimum area filter (≥ 1 ha): "
        f"{len(dissolved):,} "
        f"(removed: {n_before - len(dissolved):,})"
    )

    print("\nArea distribution of dissolved polygons (ha):")
    for p in [25, 50, 75, 90, 99]:
        print(f"  P{p:2d}: {dissolved['area_ha'].quantile(p/100):.3f} ha")
    print(f"  Max: {dissolved['area_ha'].max():.3f} ha")

    # =========================================================================
    # STEP 4: Export
    # =========================================================================
    # Filename suffix '_new_buildup_ratio1' indicates areas consisting entirely
    # of newly constructed buildings.
    out_name = "New_Housing_Development_Areas_new_buildup_ratio1_dissolved.gpkg"

    dissolved.to_file(
        output_path / out_name,
        driver="GPKG",
        layer="New_Housing_Development_Areas_new_buildup_ratio1_dissolved"
    )

    # =========================================================================
    # SUMMARY
    # =========================================================================
    print(f"\n{'=' * 60}")
    print("RESULTS")
    print(f"{'=' * 60}")
    print(f"  Total polygons: {len(dissolved):,}")
    print(f"  Total area (ha): {dissolved['area_ha'].sum():.1f}")
    print(f"\n✓ Saved: {out_name}")


if __name__ == "__main__":
    main()